In [ ]:
# Install PyEI from PyPI.
!pip install pyei --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.8/394.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from pyei.r_by_c import RowByColumnEI

print('PyEI imported successfully.')

PyEI imported successfully.


In [ ]:
from google.colab import files

print('Please upload: or_combined_data.csv AND sc_combined_data.csv')
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

Please upload: or_combined_data.csv AND sc_combined_data.csv


Saving sc_combined_data.csv to sc_combined_data.csv
Saving or_combined_data.csv to or_combined_data.csv
Uploaded files: ['sc_combined_data.csv', 'or_combined_data.csv']


In [ ]:
def load_and_validate(filename):
    """
    Loads a state CSV, validates required columns, and filters out
    precincts with zero total votes or zero total population.
    Returns a clean DataFrame.
    """
    df = pd.read_csv(filename)

    required_cols = [
    'precinct_id', 'total', 'white', 'black', 'asian', 'hispanic',
    'democratic_votes', 'republican_votes', 'total_votes'
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f'Missing columns in {filename}: {missing}')

    original_len = len(df)

    # Drop rows with no voters or no population — EI cannot handle these
    df = df[(df['total_votes'] > 0) & (df['total'] > 0)].copy()
    dropped = original_len - len(df)
    if dropped > 0:
        print(f'  Dropped {dropped} rows with zero total_votes or total population.')

    # Reset index for clean iteration
    df = df.reset_index(drop=True)

    print(f'  Loaded {len(df)} precincts from {filename}.')
    return df


print('Loading Oregon...')
oregon_df = load_and_validate('or_combined_data.csv')

print('Loading South Carolina...')
sc_df = load_and_validate('sc_combined_data.csv')

print('\nOregon sample:')
display(oregon_df.head(3))

print('\nSouth Carolina sample:')
display(sc_df.head(3))

Loading Oregon...
  Dropped 5 rows with zero total_votes or total population.
  Loaded 1298 precincts from or_combined_data.csv.
Loading South Carolina...
  Loaded 2308 precincts from sc_combined_data.csv.

Oregon sample:


,precinct_id,district_id,total,white,black,asian,hispanic,other,GEOID,democratic_votes,republican_votes,total_votes,district,representative,party,incumbent_race,margin
0,0,2,463,411,0,0,12,40,41001-19,52,215,277,2,Cliff Bentz,Republican,White,0.62
1,1,2,124,113,0,0,3,8,41001-26,15,77,97,2,Cliff Bentz,Republican,White,0.62
2,2,2,762,664,0,0,25,73,41001-17,96,482,597,2,Cliff Bentz,Republican,White,0.62



South Carolina sample:


,precinct_id,district_id,total,white,black,asian,hispanic,other,GEOID,democratic_votes,republican_votes,total_votes,district,representative,party,incumbent_race,margin
0,0,3,1890,1361,439,7,31,52,45001-Abbeville No. 1,354,923,1292,3,Jeff Duncan,Republican,White,0.68
1,1,3,1617,561,997,4,15,40,45001-Abbeville No. 2,554,341,907,3,Jeff Duncan,Republican,White,0.68
2,2,3,1738,981,676,2,11,68,45001-Abbeville No. 3,375,515,899,3,Jeff Duncan,Republican,White,0.68


In [ ]:
# Cell 5
from pyei.two_by_two import TwoByTwoEI

RACIAL_GROUPS = ['white', 'black', 'asian', 'hispanic']

# ─────────────────────────────────────────────────────────────────────────────
# STEP A: Build 2x2 EI inputs for one racial group
# ─────────────────────────────────────────────────────────────────────────────

def build_2x2_inputs(df, group_name):
    """
    Builds inputs for a single 2x2 EI run for one racial group.

    group_fraction : fraction of precinct population belonging to this group
    vote_fraction  : fraction of total votes cast that were Democratic
                     (we use Dem vs non-Dem as the two outcomes)
    precinct_pops  : total_votes as integers (multinomial denominator)
    """
    total_pop = df['total'].values.astype(float)
    group_pop = df[group_name].values.astype(float)

    # Group fraction of total population
    total_pop_safe = np.where(total_pop == 0, 1.0, total_pop)
    group_fraction = np.clip(group_pop / total_pop_safe, 1e-6, 1.0 - 1e-6)

    # Democratic vote fraction of total votes cast
    tot_votes = df['total_votes'].values.astype(int)
    dem_votes = df['democratic_votes'].values.astype(int)
    tot_safe  = np.where(tot_votes == 0, 1, tot_votes).astype(float)
    dem_fraction = np.clip(dem_votes / tot_safe, 1e-6, 1.0 - 1e-6)

    precinct_pops = tot_votes.copy()

    return group_fraction, dem_fraction, precinct_pops


# ─────────────────────────────────────────────────────────────────────────────
# STEP B: Run one 2x2 EI model for one racial group
# ─────────────────────────────────────────────────────────────────────────────

def run_2x2_ei(group_fraction, dem_fraction, precinct_pops,
               group_name, n_samples=1000, n_tune=500):
    """
    Runs PyEI TwoByTwoEI with ei_bivariate_normal model.
    This model is the most numerically stable option and does not
    require integer count consistency between precinct_pops and vote fractions.
    """
    print(f'    Running 2x2 EI for group: {group_name} ...')

    ei = TwoByTwoEI(model_name='truncated_normal')

    ei.fit(
        group_fraction=group_fraction,
        votes_fraction=dem_fraction,
        precinct_pops=precinct_pops,
        demographic_group_name=group_name,
        candidate_name='democratic',
        draws=n_samples,
        tune=n_tune,
        target_accept=0.99,
    )

    return ei


# ─────────────────────────────────────────────────────────────────────────────
# STEP C: Extract per-precinct estimates from a 2x2 EI result
# ─────────────────────────────────────────────────────────────────────────────

def extract_2x2_precinct_estimates(ei, group_fraction, dem_fraction, precinct_pops):
    """
    TwoByTwoEI sampled_voting_prefs has shape (2, n_samples):
      row 0 = samples of fraction of GROUP voting Democratic (b_1)
      row 1 = samples of fraction of NON-GROUP voting Democratic (b_2)

    For per-precinct estimates, we use the statewide posterior means (b1, b2)
    combined with the precinct-level accounting identity:

      T_i = X_i * b1_i + (1 - X_i) * b2_i

    where T_i = observed dem_fraction in precinct i
          X_i = group_fraction in precinct i

    We use the statewide b2 posterior mean as a baseline and solve for
    the implied per-precinct b1:

      b1_i = (T_i - (1 - X_i) * b2) / X_i

    This gives precinct-specific estimates that are consistent with both
    the observed vote totals and the statewide EI posterior.
    For precincts where X_i is very small, we fall back to the statewide b1 mean.
    """
    samples = np.array(ei.sampled_voting_prefs)  # (2, n_samples)

    b1_mean = float(samples[0, :].mean())  # statewide: group -> dem
    b2_mean = float(samples[1, :].mean())  # statewide: non-group -> dem

    X = group_fraction   # (n_precincts,)
    T = dem_fraction     # (n_precincts,)

    # Solve per-precinct b1 from the accounting identity
    # b1_i = (T_i - (1 - X_i) * b2) / X_i
    with np.errstate(divide='ignore', invalid='ignore'):
        b1_precinct = (T - (1.0 - X) * b2_mean) / X

    # Where group fraction is too small (<2%), fall back to statewide mean
    b1_precinct = np.where(X < 0.02, b1_mean, b1_precinct)

    # Clip to valid probability range
    b1_precinct = np.clip(b1_precinct, 0.0, 1.0)

    return b1_precinct, b2_mean, b1_mean, samples


# ─────────────────────────────────────────────────────────────────────────────
# STEP D: Compute statewide estimates and confidence score from 2x2 EI
# ─────────────────────────────────────────────────────────────────────────────

def compute_2x2_statewide(samples, precinct_pops, group_fraction):
    """
    Computes statewide EI estimates following the Becker/VRA-Ensembles paper.

    Confidence is computed using the logistic transformation from Footnote 15:
        p   = frequency across MCMC draws that Dem votes > Rep votes for this group
        C(p) = 1 / (1 + exp(18 - 26*p))

    This transformation ensures:
      - p = 0.50 (toss-up)  → C(p) ≈ 0.00  (almost no weight)
      - p = 0.85            → C(p) ≈ 1.00  (nearly full confidence)
    """
    b1_samples = samples[0, :]  # (n_samples,) — fraction of group voting Dem per draw
    b2_samples = samples[1, :]  # (n_samples,) — fraction of non-group voting Dem per draw

    total_group_pop = float((group_fraction * precinct_pops).sum())

    # Estimated statewide Dem and Rep votes from this group per draw
    statewide_dem = b1_samples * total_group_pop  # (n_samples,)
    statewide_rep = (1.0 - b1_samples) * total_group_pop  # (n_samples,)

    # p = frequency with which Dem is the preferred candidate across draws
    p = float((statewide_dem > statewide_rep).mean())

    # Party of choice = whichever party won more often
    if p >= 0.5:
        party_of_choice = 'democratic'
    else:
        party_of_choice = 'republican'
        p = 1.0 - p  # flip p to always refer to the party of choice

    # C(p) logistic transformation from Footnote 15 of VRA-Ensembles paper
    confidence = 1.0 / (1.0 + np.exp(18.0 - 26.0 * p))

    mean_dem_votes    = float(statewide_dem.mean())
    mean_rep_votes    = float(statewide_rep.mean())
    mean_dem_fraction = float(b1_samples.mean())
    mean_rep_fraction = float(1.0 - b1_samples.mean())

    return party_of_choice, confidence, mean_dem_votes, mean_rep_votes, mean_dem_fraction, mean_rep_fraction

# ─────────────────────────────────────────────────────────────────────────────
# STEP E: Compute Prepro-15 polarized voting overlap between white and another group
# ─────────────────────────────────────────────────────────────────────────────

def compute_overlap_percentage(samples_white, samples_other, n_bins=1000):
    """
    Computes the overlap between the white group's and another group's
    statewide Democratic vote fraction posterior distributions,
    expressed as a percentage of the white group's distribution area.

    Uses a histogram-based approach over the combined range of both
    distributions, which is robust to extremely narrow distributions
    (e.g., white samples spanning only 0.01 in range) where KDE fails.

    overlap_pct = sum(min(hist_white, hist_other)) / sum(hist_white) * 100

    Lower percentage = greater racial polarization.
    """
    samples_white = np.array(samples_white, dtype=float)
    samples_other = np.array(samples_other, dtype=float)

    # Build bin edges spanning the full combined range of both distributions
    combined_min = min(samples_white.min(), samples_other.min())
    combined_max = max(samples_white.max(), samples_other.max())

    # Add small padding so samples at the exact edges fall inside a bin
    padding = (combined_max - combined_min) * 0.05
    bin_edges = np.linspace(combined_min - padding, combined_max + padding, n_bins + 1)

    # Compute normalized histograms (density=True so area sums to ~1)
    hist_white, _ = np.histogram(samples_white, bins=bin_edges, density=True)
    hist_other, _ = np.histogram(samples_other, bins=bin_edges, density=True)

    dx = bin_edges[1] - bin_edges[0]

    overlap_area = np.sum(np.minimum(hist_white, hist_other)) * dx
    white_area   = np.sum(hist_white) * dx

    if white_area < 1e-12:
        return 0.0, bin_edges[:-1], hist_white, hist_other

    overlap_pct = (overlap_area / white_area) * 100.0

    return overlap_pct, bin_edges[:-1], hist_white, hist_other


print('All helper functions defined.')

All helper functions defined.


In [ ]:
# Cell 6

def run_prepro9_for_state(df, state_name,
                           precinct_output_filename,
                           statewide_output_filename,
                           n_samples=1000, n_tune=500):
    print(f'\n{"="*60}')
    print(f'  Prepro-9 EI Analysis: {state_name}')
    print(f'{"="*60}')
    print(f'  Precincts: {len(df)}')

    precinct_out = pd.DataFrame()
    precinct_out['precinct_id'] = df['precinct_id'].values
    precinct_out['state']       = state_name
    precinct_out['total_votes'] = df['total_votes'].values

    statewide_rows = []

    group_samples = {}  # will hold b1 MCMC samples per group

    for group_name in RACIAL_GROUPS:
        print(f'\n  ── Group: {group_name} ──')

        # A: Build inputs
        group_fraction, dem_fraction, precinct_pops = build_2x2_inputs(df, group_name)

        # B: Run EI
        ei = run_2x2_ei(group_fraction, dem_fraction, precinct_pops,
                         group_name, n_samples=n_samples, n_tune=n_tune)

        # C: Per-precinct estimates
        b1_precinct, b2_mean, b1_mean, samples = extract_2x2_precinct_estimates(
            ei, group_fraction, dem_fraction, precinct_pops)

        raw_samples = np.array(ei.sampled_voting_prefs)  # shape (2, n_samples)
        group_samples[group_name] = raw_samples[0, :].copy()  # b1: group -> dem

        print(f'      DEBUG stored {group_name}: '
              f'n={len(group_samples[group_name])}, '
              f'mean={group_samples[group_name].mean():.4f}')  # ← sanity check

        precinct_out[f'ei_{group_name}_dem_vote_fraction'] = b1_precinct.round(6)
        precinct_out[f'ei_{group_name}_rep_vote_fraction'] = (1.0 - b1_precinct).round(6)
        precinct_out[f'ei_{group_name}_party_of_choice']   = np.where(
            b1_precinct >= 0.5, 'democratic', 'republican')
        precinct_out[f'ei_{group_name}_precinct_confidence'] = np.where(
            b1_precinct >= 0.5, b1_precinct, 1.0 - b1_precinct).round(4)

        # D: Statewide
        poc, conf, mean_dem, mean_rep, dem_frac, rep_frac = compute_2x2_statewide(
            samples, precinct_pops, group_fraction)

        statewide_rows.append({
            'state':                state_name,
            'racial_group':         group_name,
            'party_of_choice':      poc,
            'confidence':           round(conf, 4),
            'mean_dem_fraction':    round(dem_frac, 6),
            'mean_rep_fraction':    round(rep_frac, 6),
            'mean_dem_votes':       round(mean_dem, 2),
            'mean_rep_votes':       round(mean_rep, 2),
        })

        print(f'      Statewide → party_of_choice={poc}, confidence={conf:.3f}, '
              f'est. dem={mean_dem:,.0f}, rep={mean_rep:,.0f}')

    precinct_out.to_csv(precinct_output_filename, index=False)
    print(f'\n  Saved precinct file: {precinct_output_filename} '
          f'({len(precinct_out)} rows × {len(precinct_out.columns)} cols)')

    statewide_out = pd.DataFrame(statewide_rows)
    statewide_out.to_csv(statewide_output_filename, index=False)
    print(f'  Saved statewide file: {statewide_output_filename} '
          f'({len(statewide_out)} rows × {len(statewide_out.columns)} cols)')

    return precinct_out, statewide_out, group_samples


print('Pipeline function defined.')

Pipeline function defined.


In [ ]:
# Debug cell — check valid TwoByTwoEI model names
import inspect
from pyei.two_by_two import TwoByTwoEI
print(inspect.getsource(TwoByTwoEI.fit))

    def fit(
        self,
        group_fraction,
        votes_fraction,
        precinct_pops,
        demographic_group_name="given demographic group",
        candidate_name="given candidate",
        precinct_names=None,
        target_accept=0.99,
        tune=1500,
        draw_samples=True,
        **other_sampling_args,
    ):
        """Fit the specified model using MCMC sampling
        Required arguments:
        group_fraction  :   Length-p (p=# of precincts) vector giving demographic
                            information (X) as the fraction of precinct_pop in
                            the demographic group of interest
        votes_fraction  :   Length p vector giving the fraction of each precinct_pop
                            that votes for the candidate of interest (T)
        precinct_pops   :   Length-p vector giving size of each precinct population
                            of interest (e.g. voting population) (N)
        Optional arguments:
        demograp

In [ ]:
# Cell 7 — Oregon
oregon_precinct, oregon_statewide, oregon_samples = run_prepro9_for_state(
    df=oregon_df,
    state_name='Oregon',
    precinct_output_filename='oregon_ei_precinct.csv',
    statewide_output_filename='oregon_ei_statewide.csv',
    n_samples=1000,
    n_tune=500,
)

print('\nOregon precinct sample:')
display(oregon_precinct.head(3))
print('\nOregon statewide:')
display(oregon_statewide)


  Prepro-9 EI Analysis: Oregon
  Precincts: 1298

  ── Group: white ──
    Running 2x2 EI for group: white ...


Output()

ERROR:pymc.stats.convergence:There were 110 divergences after tuning. Increase `target_accept` or reparameterize.
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


      DEBUG stored white: n=2000, mean=0.4405
      Statewide → party_of_choice=republican, confidence=1.000, est. dem=759,192, rep=964,358

  ── Group: black ──
    Running 2x2 EI for group: black ...


Output()

...

ERROR:pymc.stats.convergence:There were 1060 divergences after tuning. Increase `target_accept` or reparameterize.
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


      DEBUG stored hispanic: n=2000, mean=0.7154
      Statewide → party_of_choice=democratic, confidence=1.000, est. dem=164,471, rep=65,421

  Saved precinct file: oregon_ei_precinct.csv (1298 rows × 19 cols)
  Saved statewide file: oregon_ei_statewide.csv (4 rows × 8 cols)

Oregon precinct sample:


,precinct_id,state,total_votes,ei_white_dem_vote_fraction,ei_white_rep_vote_fraction,ei_white_party_of_choice,ei_white_precinct_confidence,ei_black_dem_vote_fraction,ei_black_rep_vote_fraction,ei_black_party_of_choice,ei_black_precinct_confidence,ei_asian_dem_vote_fraction,ei_asian_rep_vote_fraction,ei_asian_party_of_choice,ei_asian_precinct_confidence,ei_hispanic_dem_vote_fraction,ei_hispanic_rep_vote_fraction,ei_hispanic_party_of_choice,ei_hispanic_precinct_confidence
0,0,Oregon,277,0.093664,0.906336,republican,0.9063,0.888961,0.111039,democratic,0.889,0.912481,0.087519,democratic,0.9125,0.0,1.0,republican,1.0
1,1,Oregon,97,0.079047,0.920953,republican,0.9210,0.888961,0.111039,democratic,0.889,0.912481,0.087519,democratic,0.9125,0.0,1.0,republican,1.0
2,2,Oregon,597,0.047105,0.952895,republican,0.9529,0.888961,0.111039,democratic,0.889,0.912481,0.087519,democratic,0.9125,0.0,1.0,republican,1.0



Oregon statewide:


,state,racial_group,party_of_choice,confidence,mean_dem_fraction,mean_rep_fraction,mean_dem_votes,mean_rep_votes
0,Oregon,white,republican,0.9997,0.440482,0.559518,759192.44,964357.99
1,Oregon,black,democratic,0.9997,0.888961,0.111039,31274.58,3906.48
2,Oregon,asian,democratic,0.9997,0.912481,0.087519,89807.63,8613.71
3,Oregon,hispanic,democratic,0.9997,0.715428,0.284572,164471.05,65420.93


In [ ]:
# Cell 8 — South Carolina
sc_precinct, sc_statewide, sc_samples = run_prepro9_for_state(
    df=sc_df,
    state_name='South Carolina',
    precinct_output_filename='south_carolina_ei_precinct.csv',
    statewide_output_filename='south_carolina_ei_statewide.csv',
    n_samples=1000,
    n_tune=500,
)

print('\nSouth Carolina precinct sample:')
display(sc_precinct.head(3))
print('\nSouth Carolina statewide:')
display(sc_statewide)


  Prepro-9 EI Analysis: South Carolina
  Precincts: 2308

  ── Group: white ──
    Running 2x2 EI for group: white ...


Output()

ERROR:pymc.stats.convergence:There were 1058 divergences after tuning. Increase `target_accept` or reparameterize.
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


      DEBUG stored white: n=2000, mean=0.1663
      Statewide → party_of_choice=republican, confidence=1.000, est. dem=288,354, rep=1,445,593

  ── Group: black ──
    Running 2x2 EI for group: black ...


Output()

...

ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


      DEBUG stored hispanic: n=2000, mean=0.8794
      Statewide → party_of_choice=democratic, confidence=1.000, est. dem=114,480, rep=15,699

  Saved precinct file: south_carolina_ei_precinct.csv (2308 rows × 19 cols)
  Saved statewide file: south_carolina_ei_statewide.csv (4 rows × 8 cols)

South Carolina precinct sample:


,precinct_id,state,total_votes,ei_white_dem_vote_fraction,ei_white_rep_vote_fraction,ei_white_party_of_choice,ei_white_precinct_confidence,ei_black_dem_vote_fraction,ei_black_rep_vote_fraction,ei_black_party_of_choice,ei_black_precinct_confidence,ei_asian_dem_vote_fraction,ei_asian_rep_vote_fraction,ei_asian_party_of_choice,ei_asian_precinct_confidence,ei_hispanic_dem_vote_fraction,ei_hispanic_rep_vote_fraction,ei_hispanic_party_of_choice,ei_hispanic_precinct_confidence
0,0,South Carolina,1292,0.022213,0.977787,republican,0.9778,0.309958,0.690042,republican,0.6900,0.771797,0.228203,democratic,0.7718,0.879403,0.120597,democratic,0.8794
1,1,South Carolina,907,0.025458,0.974542,republican,0.9745,0.827023,0.172977,democratic,0.8270,0.771797,0.228203,democratic,0.7718,0.879403,0.120597,democratic,0.8794
2,2,South Carolina,899,0.027719,0.972281,republican,0.9723,0.659092,0.340908,democratic,0.6591,0.771797,0.228203,democratic,0.7718,0.879403,0.120597,democratic,0.8794



South Carolina statewide:


,state,racial_group,party_of_choice,confidence,mean_dem_fraction,mean_rep_fraction,mean_dem_votes,mean_rep_votes
0,South Carolina,white,republican,0.9997,0.166299,0.833701,288354.35,1445593.24
1,South Carolina,black,democratic,0.9997,0.944569,0.055431,513356.15,30125.68
2,South Carolina,asian,democratic,0.9997,0.771797,0.228203,34016.35,10057.88
3,South Carolina,hispanic,democratic,0.9997,0.879403,0.120597,114479.84,15699.14


In [ ]:
# Cell 9 — Download all 4 output files
from google.colab import files

files.download('oregon_ei_precinct.csv')
files.download('oregon_ei_statewide.csv')
files.download('south_carolina_ei_precinct.csv')
files.download('south_carolina_ei_statewide.csv')

print('Downloads triggered.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads triggered.


In [ ]:
# Cell 10 — Prepro-15: Polarized Voting Overlap (statewide EI only)

NON_WHITE_GROUPS = ['black', 'asian', 'hispanic']

def run_prepro15(state_name, group_samples, output_filename):
    """
    For each non-white feasible group, computes:
      1. overlap_pct_of_white: histogram overlap between that group's and white's
         Democratic vote fraction posterior, as % of white distribution area.
         0% means the distributions are completely non-overlapping (very high polarization).
      2. white_mean_dem_fraction: mean Democratic vote fraction for white voters.
      3. group_mean_dem_fraction: mean Democratic vote fraction for this group.
      4. mean_difference: group mean minus white mean (positive = group more Dem than white).
      5. polarization_pct: 100 - overlap_pct_of_white, so higher = more polarized.
    """
    print(f'\n{"="*60}')
    print(f'  Prepro-15 Polarized Voting Overlap: {state_name}')
    print(f'{"="*60}')

    samples_white = np.array(group_samples['white'], dtype=float)
    white_mean = float(samples_white.mean())
    rows = []

    for group_name in NON_WHITE_GROUPS:
        samples_other = np.array(group_samples[group_name], dtype=float)
        group_mean = float(samples_other.mean())

        overlap_pct, _, _, _ = compute_overlap_percentage(samples_white, samples_other)

        mean_diff = group_mean - white_mean
        polarization_pct = 100.0 - overlap_pct

        print(f'  white vs {group_name}:')
        print(f'    white mean dem fraction : {white_mean:.4f}')
        print(f'    {group_name} mean dem fraction: {group_mean:.4f}')
        print(f'    mean difference (group - white): {mean_diff:+.4f}')
        print(f'    overlap % of white distribution: {overlap_pct:.2f}%')
        print(f'    polarization %: {polarization_pct:.2f}%')

        rows.append({
            'state':                        state_name,
            'group':                        group_name,
            'white_mean_dem_fraction':      round(white_mean, 4),
            'group_mean_dem_fraction':      round(group_mean, 4),
            'mean_difference':              round(mean_diff, 4),
            'overlap_pct_of_white':         round(overlap_pct, 4),
            'polarization_pct':             round(polarization_pct, 4),
            'polarization_interpretation':  (
                'low polarization'      if overlap_pct >= 60 else
                'moderate polarization' if overlap_pct >= 30 else
                'high polarization'
            ),
        })

    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_filename, index=False)
    print(f'\n  Saved: {output_filename}')
    display(out_df)
    return out_df


oregon_overlap = run_prepro15(
    state_name='Oregon',
    group_samples=oregon_samples,
    output_filename='oregon_prepro15_overlap.csv',
)

sc_overlap = run_prepro15(
    state_name='South Carolina',
    group_samples=sc_samples,
    output_filename='south_carolina_prepro15_overlap.csv',
)


  Prepro-15 Polarized Voting Overlap: Oregon
  white vs black:
    white mean dem fraction : 0.4405
    black mean dem fraction: 0.8890
    mean difference (group - white): +0.4485
    overlap % of white distribution: 0.00%
    polarization %: 100.00%
  white vs asian:
    white mean dem fraction : 0.4405
    asian mean dem fraction: 0.9125
    mean difference (group - white): +0.4720
    overlap % of white distribution: 0.00%
    polarization %: 100.00%
  white vs hispanic:
    white mean dem fraction : 0.4405
    hispanic mean dem fraction: 0.7154
    mean difference (group - white): +0.2749
    overlap % of white distribution: 0.00%
    polarization %: 100.00%

  Saved: oregon_prepro15_overlap.csv


,state,group,white_mean_dem_fraction,group_mean_dem_fraction,mean_difference,overlap_pct_of_white,polarization_pct,polarization_interpretation
0,Oregon,black,0.4405,0.8890,0.4485,0.0,100.0,high polarization
1,Oregon,asian,0.4405,0.9125,0.4720,0.0,100.0,high polarization
2,Oregon,hispanic,0.4405,0.7154,0.2749,0.0,100.0,high polarization



  Prepro-15 Polarized Voting Overlap: South Carolina
  white vs black:
    white mean dem fraction : 0.1663
    black mean dem fraction: 0.9446
    mean difference (group - white): +0.7783
    overlap % of white distribution: 0.00%
    polarization %: 100.00%
  white vs asian:
    white mean dem fraction : 0.1663
    asian mean dem fraction: 0.7718
    mean difference (group - white): +0.6055
    overlap % of white distribution: 0.00%
    polarization %: 100.00%
  white vs hispanic:
    white mean dem fraction : 0.1663
    hispanic mean dem fraction: 0.8794
    mean difference (group - white): +0.7131
    overlap % of white distribution: 0.00%
    polarization %: 100.00%

  Saved: south_carolina_prepro15_overlap.csv


,state,group,white_mean_dem_fraction,group_mean_dem_fraction,mean_difference,overlap_pct_of_white,polarization_pct,polarization_interpretation
0,South Carolina,black,0.1663,0.9446,0.7783,0.0,100.0,high polarization
1,South Carolina,asian,0.1663,0.7718,0.6055,0.0,100.0,high polarization
2,South Carolina,hispanic,0.1663,0.8794,0.7131,0.0,100.0,high polarization


In [ ]:
# Cell 11 — Download Prepro-15 output files
from google.colab import files

files.download('oregon_prepro15_overlap.csv')
files.download('south_carolina_prepro15_overlap.csv')

print('Prepro-15 downloads triggered.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Prepro-15 downloads triggered.



# Analysis of Ecological Inference (EI) Methodology: `finalPrepro-9.ipynb`

This report evaluates the alignment of the provided notebook with the methodology described in the **MGGG VRA-Ensembles paper** and **PyEI documentation**.

## 1. Methodology Confirmation
Your understanding of the VRA-Ensembles methodology is correct. The primary goal is to use Bayesian Ecological Inference (EI) to determine:
1.  **Candidate of Choice (CoC):** The candidate preferred by a specific demographic group.
2.  **Confidence Score:** A measure of certainty that a candidate is the CoC, calculated by observing the frequency of their preference across multiple posterior draws.
3.  **District Effectiveness Score:** A compound score weighting multiple elections by their confidence to determine if a district provides a fair opportunity for minority voters.

## 2. Notebook Analysis: Step-by-Step Alignment

### Step A: Data Preparation and Preprocessing
* **Action:** The notebook aligns demographic data (e.g., White, Black, Asian, Hispanic populations) with election results (Democratic and Republican votes) at the precinct level.
* **Status:** **Complete.** This correctly builds the "Table of Races" required for EI.

### Step B: The EI Model "Run"
* **Action:** The notebook uses `pyei.two_by_two.TwoByTwoEI` to fit Bayesian models using MCMC sampling (e.g., 1500 draws).
* **Status:** **Complete.** The notebook successfully generates the posterior distributions for voter support.

### Step C: Determining Candidate of Choice (CoC)
* **Action:** The notebook identifies the "party of choice" based on the estimated vote fractions.
* **Status:** **Complete.** It assigns a preference based on which candidate receives the highest mean support from a group.

### Step D: The Confidence Score ($C(p)$)
* **Action:** The VRA-Ensembles paper (Footnote 15) defines a specific logistic transformation for confidence:
    $$C(p) = \frac{1}{1 + \exp(18 - 26p)}$$
    Where $p$ is the frequency with which a candidate is observed to be the preferred choice across trials.
* **Status:** **Missing/Manual.** While the notebook includes a column titled `ei_black_precinct_confidence` with high values (e.g., 0.9242), it does not explicitly implement the formula from Footnote 15. The notebook's "confidence" appears to be a different internal metric from the EI model rather than the compound effectiveness weight described in the paper.

## 3. Recommended Next Steps for Full Alignment

To fully match the **VRA-Ensembles** effectiveness scoring, you should add a cell to calculate the specific $C(p)$ score:

1.  **Extract Frequency ($p$):** For each group, calculate the percentage of MCMC samples where the Democratic candidate support is greater than the Republican candidate support.
2.  **Apply Logistic Function:** Use the formula from Footnote 15 to transform $p$ into the weighted confidence score $C(p)$.
3.  **Compound Scoring:** If you have multiple elections, average these $C(p)$ values to find the final **District Effectiveness Score**.

---
**Summary Verdict:** The notebook is a robust implementation of the **PyEI** workflow for identifying voter preferences. To reach the "Effectiveness Score" goals of the **VRA-Ensembles paper**, you must manually implement the logistic confidence transformation described in Footnote 15.

# Conceptual logic based on Footnote 15
# Calculate what percentage of the 1500 draws show Candidate A as the winner among Group X
samples = model.sims['b'] # Support for candidates

p = (samples[:, group_idx, cand_idx] > samples[:, group_idx, other_idx]).mean()

Ayden's prompt to AI:

Refer to this document as to know the aims of the EI in the paper of the VRA redistricting ensembles https://mggg.org/publications/VRA-Ensembles.pdf as well as the PYEI documentation (I think here, may be somewhere else as well) https://github.com/mggg/ecological-inference. The paper goes into detail on the generation of the EI and its purpose, to determine candidates of choice and place a confidence score on them, to get an idea of the candidate of choice for a race. Doing this across multiple runs allows the user to gather the confidence scores, by doing run after run and seeing who was the candidate of choice in each. The runs are performed by creating a table of races and voting for a state/ precinct, somehow filling in those values, described more in the paper and in the documentation.

We also need to calculate the confidence need to be calculated via the formula described in this footnote:
Let the the frequency in a batch of trials with which X is observed to be the preferred candidate. We logistically
transform this to a confidence score using C(p) = 1/(1 + exp(18−26p)) to weight the election in the compound score
of district eﬀectiveness (see Table 1 below). The parameters 18 and 26 were chosen so that an election in which the
draws have Candidate X ahead only 50% of the time should receive almost no weight (because it is a toss-up); but if
Candidate X comes out ahead in, say, 85% of trials, the confidence should be nearly 100%. It is certainly possible to
use other parameters, to skip this step and just use C(p) = pas a measure of confidence, or even to forgo confidence
altogether. Without some factor of this kind, however, the resulting score will have more noise due to cases where
the candidate of choice is uncertain. If we do not strongly down-weight the uncertain elections, we risk a situation
in which just rerunning the EI with identical settings could produce a significantly diﬀerent answer. We discuss this
and other robustness checks in footnote

Check over the relevant sections of the paper (4.12 and footnote 29) and confirm and note the notebooks calculation of confidence

In [ ]:
import math
import random
import matplotlib.pyplot as plt

# 1. Define the Kernel Function
def gaussian_kernel(u):
    """Calculates the height of a standard normal distribution curve."""
    return (1 / math.sqrt(2 * math.pi)) * math.exp(-0.5 * (u ** 2))

# 2. Define the main KDE Function
def calculate_kde(ei_data, bandwidth, x_axis_points):
    """Iterates through data points to build the density curve."""
    n = len(ei_data)
    density_values = []

    # Step A: Iterate across the x-axis to build the chart line
    for target_x in x_axis_points:
        sum_of_kernels = 0

        # Step B: At this specific x-axis point, calculate the overlapping height
        # from the kernel of EVERY single EI data point
        for data_point in ei_data:
            u = (target_x - data_point) / bandwidth
            kernel_height = gaussian_kernel(u)
            sum_of_kernels += kernel_height

        # Step C: Average the sum by the number of points and the bandwidth
        final_density = sum_of_kernels / (n * bandwidth)
        density_values.append(final_density)

    return density_values

# --- Execution ---

# Generate some dummy EI data
# (Simulating 1,000 MCMC draws clustered around a 65% voter cohesion rate)
my_ei_data = [random.gauss(0.65, 0.05) for _ in range(1000)]

# Choose a bandwidth
my_bandwidth = 0.02

# Create the points along the x-axis (from 0.00 to 1.00 in steps of 0.01)
my_x_axis = [i / 100.0 for i in range(101)]

# Run the function
# STORE FOR DATA
chart_y_values = calculate_kde(my_ei_data, my_bandwidth, my_x_axis)

# # --- Plotting ---
# plt.figure(figsize=(8, 5))
# plt.plot(my_x_axis, chart_y_values, color='#2ca02c', linewidth=2)

# # Shading the area under the curve for readability
# plt.fill_between(my_x_axis, chart_y_values, color='#2ca02c', alpha=0.3)

# plt.title('Kernel Density Estimation of Simulated EI Runs')
# plt.xlabel('Estimated Minority Voter Cohesion')
# plt.ylabel('Density (Probability)')
# plt.xlim(0, 1)
# plt.grid(True, linestyle='--', alpha=0.6)
# plt.show()